In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
df = pd.read_csv('synthetic_injection_molding.csv')
df.drop(columns=['timestamp'], inplace=True)
df.head(1)

,mold_name,material_name,machine_tonnage,cycle_number,hopper_temp,nozzle_temp,H1,H2,H3,mold_core_temp,...,total_cycle_time,Hopper_State,Barrel_Heater_State,Screw_State,Injection_Unit_State,Hydraulic_System_State,Clamp_Unit_State,Mold_State,Cooling_System_State,Ejector_System_State
0,Smart Router Enclosure,"ABS (Flame Retardant, Medium Flow)",180,1,45,230,225,211,201,67,...,24.61,0,0,0,0,0,0,0,0,0


In [3]:
target = ['Hopper_State','Barrel_Heater_State', 'Screw_State', 'Injection_Unit_State','Hydraulic_System_State', 
               'Clamp_Unit_State', 'Mold_State','Cooling_System_State', 'Ejector_System_State']  
sensor = [i for i in df.columns if i not in target]
sensor.remove('mold_name')
sensor.remove('material_name')
len(sensor)

34

In [4]:
sensor

['machine_tonnage',
 'cycle_number',
 'hopper_temp',
 'nozzle_temp',
 'H1',
 'H2',
 'H3',
 'mold_core_temp',
 'mold_cavity_temp',
 'inj_speed1',
 'inj_speed2',
 'inj_speed3',
 'actual_inj_pressure',
 'vp_transition',
 'inj_time_actual',
 'holding_pressure_stage1',
 'holding_pressure_stage2',
 'holding_pressure_stage3',
 'screw_rpm',
 'back_pressure',
 'decompression_speed',
 'decompression_distance',
 'cushion_size',
 'clamp_force',
 'mold_close_speed',
 'mold_protection_pressure',
 'ejection_stroke',
 'ejection_speed',
 'cooling_time',
 'mold_open_reset_time',
 'part_weight',
 'shot_weight',
 'screw_diameter',
 'total_cycle_time']

In [5]:
def find_correlation(corr, threshold=0.95):
    """Iteratively drop the more globally-redundant column of the most-correlated
    remaining pair, until no pair exceeds threshold. Avoids over-pruning chains."""
    corr = corr.abs().copy()
    vals = corr.values.copy()
    np.fill_diagonal(vals, 0)  # ignore self-correlation
    corr = pd.DataFrame(vals, index=corr.index, columns=corr.columns)
    to_drop = []
    while True:
        max_corr = corr.values.max()
        if max_corr <= threshold or corr.shape[0] <= 1:
            break
        i, j = np.unravel_index(np.argmax(corr.values), corr.shape)
        col_i, col_j = corr.index[i], corr.columns[j]
        # drop whichever has the higher average correlation with everything else remaining
        drop_col = col_i if corr[col_i].mean() > corr[col_j].mean() else col_j
        to_drop.append(drop_col)
        corr = corr.drop(index=drop_col, columns=drop_col)
    return to_drop
 
# correlation-based pruning of redundant raw sensor columns
corr = df[sensor].corr()
to_drop = find_correlation(corr, threshold=0.95)
kept_numeric_cols = [c for c in sensor if c not in to_drop]
df.drop(columns=to_drop, inplace=True)
print(f"Dropped {len(to_drop)} redundant (|r|>0.95) columns, kept {len(kept_numeric_cols)}:")
print(kept_numeric_cols)

Dropped 18 redundant (|r|>0.95) columns, kept 16:
['cycle_number', 'hopper_temp', 'H1', 'mold_cavity_temp', 'inj_speed1', 'actual_inj_pressure', 'vp_transition', 'holding_pressure_stage2', 'back_pressure', 'decompression_speed', 'decompression_distance', 'cushion_size', 'clamp_force', 'mold_protection_pressure', 'ejection_stroke', 'part_weight']


## Train Test Split

In [6]:
# train: 70%, validation: 15%, test: 15%
#splits based on mold type to include all variations in data
train_list = []
val_list = []
test_list = []

for mold_name, group in df.groupby('mold_name'):
 
    n = len(group)
 
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
 
    train_list.append(group.iloc[:train_end])
    val_list.append(group.iloc[train_end:val_end])
    test_list.append(group.iloc[val_end:])
 
train_df = pd.concat(train_list).reset_index(drop=True)
val_df = pd.concat(val_list).reset_index(drop=True)
test_df = pd.concat(test_list).reset_index(drop=True)
 
print(f"Train: {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test: {test_df.shape}")


Train: (28000, 27)
Validation: (6000, 27)
Test: (6000, 27)


In [7]:
# scaling numeric features
scaler = StandardScaler()
id_cat_cols = ['mold_name', 'material_name']
mold_dummy_cols = [c for c in train_df.columns if c.startswith('mold_name_')]
 
exclude_cols = id_cat_cols + target + mold_dummy_cols
 
numeric_cols = train_df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
 
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
val_df[numeric_cols] = scaler.transform(val_df[numeric_cols])
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

In [8]:
#ohe on mold id
def add_mold_dummies(data):
    enc = pd.get_dummies(data['mold_name'], dtype=int)
    return pd.concat([enc,data], axis=1)

train_df = add_mold_dummies(train_df)
val_df = add_mold_dummies(val_df)
test_df = add_mold_dummies(test_df)

## Model Training

In [9]:
## LOGISTIC REGRESSION

drop_cols = target + ['mold_name', 'material_name']
sensors = [c for c in train_df.columns if c not in drop_cols] 
x_train, x_val, x_test = train_df[sensors], val_df[sensors], test_df[sensors]
y_train, y_val, y_test = train_df[target], val_df[target], test_df[target]
models = {}
predictions = {}

for col in target:
    model = LogisticRegression(
        solver='lbfgs',
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    )
    model.fit(x_train, y_train[col])
    models[col] = model
    predictions[col] = model.predict(x_val)

# Evaluation
res = []
for col in target:
    y_true = y_val[col]
    y_pred = predictions[col]
    res.append({
        'Target': col,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1_macro': f1_score(y_true,y_pred,average='macro',zero_division=0),
        'QWK': cohen_kappa_score(y_true,y_pred,weights='quadratic')
    })

res_df = pd.DataFrame(res)
print(res_df.round(3).to_string(index=False))
print(
    '\nMEAN Accuracy: %.3f | MEAN F1_macro: %.3f | MEAN QWK: %.3f'
    % (
        res_df['Accuracy'].mean(),
        res_df['F1_macro'].mean(),
        res_df['QWK'].mean()
    )
)

                Target  Accuracy  F1_macro   QWK
          Hopper_State     0.803     0.696 0.740
   Barrel_Heater_State     0.846     0.440 0.504
           Screw_State     0.878     0.798 0.809
  Injection_Unit_State     0.823     0.745 0.658
Hydraulic_System_State     0.720     0.524 0.410
      Clamp_Unit_State     0.811     0.632 0.473
            Mold_State     0.706     0.401 0.363
  Cooling_System_State     0.662     0.441 0.265
  Ejector_System_State     0.790     0.709 0.717

MEAN Accuracy: 0.782 | MEAN F1_macro: 0.598 | MEAN QWK: 0.549


In [11]:
## RANDOM FOREST CLASSIFIER

rf = RandomForestClassifier(n_estimators=300,max_depth=10,min_samples_split=5,min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=-1)
rf.fit(x_train, y_train)

# Validation
y_pred = rf.predict(x_val)

res = []
for i, col in enumerate(target):
    res.append({
        'Target': col,
        'Accuracy': accuracy_score(y_val[col], y_pred[:, i]),
        'F1_macro': f1_score(y_val[col], y_pred[:, i], average='macro', zero_division=0),
        'QWK': cohen_kappa_score(y_val[col], y_pred[:, i], weights='quadratic')
    })

res_df = pd.DataFrame(res)
print("VALIDATION RESULTS")
print(res_df.round(3).to_string(index=False))
print('\nMEAN F1_macro: %.3f | QWK: %.3f' %
      (res_df.F1_macro.mean(), res_df.QWK.mean()))

# Test
y_pred_test = rf.predict(x_test)

test_true = y_test[target].values.flatten()
test_pred = y_pred_test.flatten()

print("\nTEST RESULTS")
print("Accuracy: %.3f" % accuracy_score(test_true, test_pred))
print("F1_macro: %.3f" % f1_score(test_true, test_pred, average='macro', zero_division=0))
print("QWK: %.3f" % cohen_kappa_score(test_true, test_pred, weights='quadratic'))

print("\nExample prediction:", y_pred_test[0])

VALIDATION RESULTS
                Target  Accuracy  F1_macro    QWK
          Hopper_State     0.534     0.368  0.067
   Barrel_Heater_State     0.886     0.364  0.066
           Screw_State     0.770     0.632  0.536
  Injection_Unit_State     0.872     0.647  0.596
Hydraulic_System_State     0.845     0.314 -0.049
      Clamp_Unit_State     0.858     0.308  0.000
            Mold_State     0.827     0.332  0.074
  Cooling_System_State     0.784     0.392  0.173
  Ejector_System_State     0.685     0.323  0.073

MEAN F1_macro: 0.409 | QWK: 0.171

TEST RESULTS
Accuracy: 0.750
F1_macro: 0.408
QWK: 0.208

Example prediction: [0 0 1 0 0 0 0 0 0]


In [10]:
## GRADIENT BOOSTING CLASSIFIER

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
import numpy as np
import pandas as pd

models = []
predictions = []

for i, col in enumerate(target):
    sample_weights = compute_sample_weight(
        class_weight='balanced',
        y=y_train[col]
    )

    model = GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    model.fit(x_train, y_train[col], sample_weight=sample_weights)
    models.append(model)
    predictions.append(model.predict(x_val))

y_pred = np.column_stack(predictions)

res = []
for i, col in enumerate(target):
    res.append({
        'Target': col,
        'Accuracy': accuracy_score(y_val[col], y_pred[:, i]),
        'F1_macro': f1_score(y_val[col], y_pred[:, i], average='macro', zero_division=0),
        'QWK': cohen_kappa_score(y_val[col], y_pred[:, i], weights='quadratic')
    })

res_df = pd.DataFrame(res)
print("VALIDATION RESULTS")
print(res_df.round(3).to_string(index=False))
print('\nMEAN F1_macro: %.3f | QWK: %.3f' %
      (res_df.F1_macro.mean(), res_df.QWK.mean()))

test_predictions = []
for model in models:
    test_predictions.append(model.predict(x_test))

y_pred_test = np.column_stack(test_predictions)

test_true = y_test[target].values.flatten()
test_pred = y_pred_test.flatten()

print("\nTEST RESULTS")
print("Accuracy: %.3f" % accuracy_score(test_true, test_pred))
print("F1_macro: %.3f" % f1_score(test_true, test_pred, average='macro', zero_division=0))
print("QWK: %.3f" % cohen_kappa_score(test_true, test_pred, weights='quadratic'))
print("\nExample prediction:", y_pred_test[0])

VALIDATION RESULTS
                Target  Accuracy  F1_macro   QWK
          Hopper_State     0.729     0.488 0.421
   Barrel_Heater_State     0.896     0.419 0.246
           Screw_State     0.571     0.528 0.415
  Injection_Unit_State     0.875     0.588 0.693
Hydraulic_System_State     0.872     0.488 0.483
      Clamp_Unit_State     0.815     0.456 0.479
            Mold_State     0.806     0.426 0.180
  Cooling_System_State     0.794     0.345 0.292
  Ejector_System_State     0.647     0.472 0.299

MEAN F1_macro: 0.468 | QWK: 0.390

TEST RESULTS
Accuracy: 0.752
F1_macro: 0.507
QWK: 0.436

Example prediction: [0 0 1 2 0 1 0 0 0]


In [ ]:
## XGBOOST CLASSIFIER

xgb = MultiOutputClassifier(
    XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softmax',
        num_class=3,
        eval_metric='mlogloss',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
)

xgb.fit(x_train, y_train)

# Validation
y_pred = xgb.predict(x_val)

res = []
for i, col in enumerate(target):
    res.append({
        'Target': col,
        'Accuracy': accuracy_score(y_val[col], y_pred[:, i]),
        'F1_macro': f1_score(y_val[col], y_pred[:, i], average='macro', zero_division=0),
        'QWK': cohen_kappa_score(y_val[col], y_pred[:, i], weights='quadratic')
    })

res_df = pd.DataFrame(res)
print("VALIDATION RESULTS")
print(res_df.round(3).to_string(index=False))
print('\nMEAN F1_macro: %.3f | QWK: %.3f' %
      (res_df.F1_macro.mean(), res_df.QWK.mean()))

# Test
y_pred_test = xgb.predict(x_test)

test_true = y_test[target].values.flatten()
test_pred = y_pred_test.flatten()

print("\nTEST RESULTS")
print("Accuracy: %.3f" % accuracy_score(test_true, test_pred))
print("F1_macro: %.3f" % f1_score(test_true, test_pred, average='macro', zero_division=0))
print("QWK: %.3f" % cohen_kappa_score(test_true, test_pred, weights='quadratic'))

print("\nExample prediction:", y_pred_test[0])

VALIDATION RESULTS
                Target  Accuracy  F1_macro   QWK
          Hopper_State     0.658     0.298 0.139
   Barrel_Heater_State     0.894     0.315 0.000
           Screw_State     0.630     0.558 0.416
  Injection_Unit_State     0.867     0.593 0.617
Hydraulic_System_State     0.902     0.495 0.567
      Clamp_Unit_State     0.853     0.395 0.283
            Mold_State     0.781     0.371 0.122
  Cooling_System_State     0.799     0.355 0.198
  Ejector_System_State     0.691     0.448 0.399

MEAN F1_macro: 0.425 | QWK: 0.305

TEST RESULTS
Accuracy: 0.753
F1_macro: 0.425
QWK: 0.293

Example prediction: [2 0 1 0 0 0 0 0 0]


In [13]:
## LIGHT GBM
models = {}
val_preds = {}
test_preds = []

for col in target:
    model = LGBMClassifier(
        objective='multiclass',
        num_class=3,
        n_estimators=300,
        num_leaves=31,
        learning_rate=0.05,
        feature_fraction=0.9,
        bagging_fraction=0.8,
        bagging_freq=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(x_train, y_train[col])
    models[col] = model
    val_preds[col] = model.predict(x_val)

# Validation results
res = []
for col in target:
    y_pred = val_preds[col]
    res.append({
        'Target': col,
        'F1_macro': f1_score(y_val[col], y_pred, average='macro', zero_division=0),
        'QWK': cohen_kappa_score(y_val[col], y_pred, weights='quadratic')
    })

res_df = pd.DataFrame(res)
print(res_df.round(3).to_string(index=False))
print('\nMEAN F1_macro: %.3f | QWK: %.3f' %
      (res_df.F1_macro.mean(), res_df.QWK.mean()))

# Test predictions
for col in target:
    test_preds.append(models[col].predict(x_test))

test_preds = np.column_stack(test_preds)

# Overall test performance
y_true = y_test[target].values.flatten()
y_pred = test_preds.flatten()

print('\nTEST RESULTS')
print('F1_macro: %.3f' % f1_score(y_true, y_pred, average='macro', zero_division=0))
print('QWK: %.3f' % cohen_kappa_score(y_true, y_pred, weights='quadratic'))

print('\nExample prediction:', test_preds[0])

                Target  F1_macro   QWK
          Hopper_State     0.320 0.244
   Barrel_Heater_State     0.365 0.067
           Screw_State     0.575 0.483
  Injection_Unit_State     0.559 0.643
Hydraulic_System_State     0.519 0.641
      Clamp_Unit_State     0.424 0.364
            Mold_State     0.341 0.032
  Cooling_System_State     0.313 0.068
  Ejector_System_State     0.385 0.291

MEAN F1_macro: 0.422 | QWK: 0.315

TEST RESULTS
F1_macro: 0.457
QWK: 0.348

Example prediction: [0 0 1 0 0 0 0 0 0]


In [14]:
## CATBOOST

models = {}
val_preds = {}
test_preds = []

for col in target:
    model = CatBoostClassifier(
        loss_function='MultiClass',
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=False,
        thread_count=-1,
        auto_class_weights='Balanced'
    )

    model.fit(x_train, y_train[col])
    models[col] = model
    val_preds[col] = model.predict(x_val).flatten().astype(int)

# Validation results
res = []
for col in target:
    y_pred = val_preds[col]
    res.append({
        'Target': col,
        'Accuracy': accuracy_score(y_val[col], y_pred),
        'F1_macro': f1_score(y_val[col], y_pred, average='macro', zero_division=0),
        'QWK': cohen_kappa_score(y_val[col], y_pred, weights='quadratic')
    })

res_df = pd.DataFrame(res)
print(res_df.round(3).to_string(index=False))
print('\nMEAN F1_macro: %.3f | QWK: %.3f' %
      (res_df.F1_macro.mean(), res_df.QWK.mean()))

# Test predictions
for col in target:
    test_preds.append(models[col].predict(x_test).flatten().astype(int))

test_preds = np.column_stack(test_preds)

# Overall test performance
y_true = y_test[target].values.flatten()
y_pred = test_preds.flatten()

print('\nTEST RESULTS')
print('F1_macro: %.3f' % f1_score(y_true, y_pred, average='macro', zero_division=0))
print('QWK: %.3f' % cohen_kappa_score(y_true, y_pred, weights='quadratic'))

print('\nExample prediction:', test_preds[0])

                Target  Accuracy  F1_macro   QWK
          Hopper_State     0.723     0.472 0.519
   Barrel_Heater_State     0.898     0.463 0.496
           Screw_State     0.763     0.674 0.628
  Injection_Unit_State     0.871     0.649 0.660
Hydraulic_System_State     0.825     0.496 0.476
      Clamp_Unit_State     0.843     0.356 0.159
            Mold_State     0.822     0.389 0.182
  Cooling_System_State     0.796     0.441 0.531
  Ejector_System_State     0.740     0.571 0.530

MEAN F1_macro: 0.501 | QWK: 0.464

TEST RESULTS
F1_macro: 0.522
QWK: 0.537

Example prediction: [0 0 1 0 0 0 0 0 0]
